In [1]:
from collections import defaultdict
from sklearn import linear_model
import numpy
import math
import gzip
import json
import dateutil.parser
import random

In [2]:
### Question 1

In [3]:
f = gzip.open('/Users/hrithik/Desktop/UC San Diego/Courses/UCSD CSE 258/Homework 1/fantasy_10000.json.gz')
dataset = []
for l in f:
    dataset.append(json.loads(l))

In [4]:
dataset[0]

{'user_id': '8842281e1d1347389f2ab93d60773d4d',
 'book_id': '18245960',
 'review_id': 'dfdbb7b0eb5a7e4c26d59a937e2e5feb',
 'rating': 5,
 'review_text': 'This is a special book. It started slow for about the first third, then in the middle third it started to get interesting, then the last third blew my mind. This is what I love about good science fiction - it pushes your thinking about where things can go. \n It is a 2015 Hugo winner, and translated from its original Chinese, which made it interesting in just a different way from most things I\'ve read. For instance the intermixing of Chinese revolutionary history - how they kept accusing people of being "reactionaries", etc. \n It is a book about science, and aliens. The science described in the book is impressive - its a book grounded in physics and pretty accurate as far as I could tell. Though when it got to folding protons into 8 dimensions I think he was just making stuff up - interesting to think about though. \n But what would 

In [5]:
def getMaxLen(dataset):
    # Find the longest review (number of characters)
    maxLen = 0
    for review in dataset:
        sample_len = len(review['review_text'])
        maxLen = max(maxLen, sample_len)

    return maxLen 

In [6]:
dataset[0]

{'user_id': '8842281e1d1347389f2ab93d60773d4d',
 'book_id': '18245960',
 'review_id': 'dfdbb7b0eb5a7e4c26d59a937e2e5feb',
 'rating': 5,
 'review_text': 'This is a special book. It started slow for about the first third, then in the middle third it started to get interesting, then the last third blew my mind. This is what I love about good science fiction - it pushes your thinking about where things can go. \n It is a 2015 Hugo winner, and translated from its original Chinese, which made it interesting in just a different way from most things I\'ve read. For instance the intermixing of Chinese revolutionary history - how they kept accusing people of being "reactionaries", etc. \n It is a book about science, and aliens. The science described in the book is impressive - its a book grounded in physics and pretty accurate as far as I could tell. Though when it got to folding protons into 8 dimensions I think he was just making stuff up - interesting to think about though. \n But what would 

In [7]:
type(dataset[0]['rating'])

int

In [8]:
maxLen = getMaxLen(dataset)
print(maxLen)

14306


In [9]:
def featureQ1(datum, maxLen):
    # Feature vector for one data point 
    return [1, len(datum['review_text'])/maxLen]

In [10]:
def Q1(dataset):
    # Implement...
    maxLen = getMaxLen(dataset)
    x = []
    y = []
    for datum in dataset:
        x.append(featureQ1(datum, maxLen))
        y.append(datum['rating'])
    # print(x[0], y[0])
    reg = linear_model.LinearRegression(fit_intercept=False)
    reg.fit(x,y)
    # yi = reg.predict(x)
    yi = numpy.sum(numpy.asarray(x) * reg.coef_, axis=1)
    # print(yi[0])
    MSE = sum((yi - y)**2)/len(y)
    theta = reg.coef_
    return theta, MSE

In [11]:
theta, MSE = Q1(dataset)
print(theta, MSE)

[3.68568136 0.98335392] 1.5522086622355356


In [12]:
### Question 2

In [13]:
def featureQ2(datum, maxLen):
    # Implement (should be 1, length feature, day feature, month feature)
    date = dateutil.parser.parse(datum['date_added'])
    weekday_features = [int(date.isoweekday() == day) for day in range(2, 8)]
    month_features = [int(date.month == month) for month in range(2, 13)]

    return [1, len(datum['review_text']) / maxLen] + weekday_features + month_features
    


In [14]:
def Q2(dataset):
    # Implement (note MSE should be a *number*, not e.g. an array of length 1)
        maxLen = getMaxLen(dataset)
        X2 = []
        Y2 = []
        for datum in dataset:
            X2.append(featureQ2(datum, maxLen))
            Y2.append(datum['rating'])
        print(X2[0], Y2[0])
        reg = linear_model.LinearRegression(fit_intercept=False)
        reg.fit(X2,Y2)
   
        Yi = numpy.sum(numpy.asarray(X2) * reg.coef_, axis=1)
   
        MSE2 = sum((Yi - Y2)**2)/len(Y2)
        # theta = reg.coef_
        return X2, Y2, MSE2

In [15]:
X2, Y2, MSE2 = Q2(dataset)
print(MSE2)

[1, 0.14581294561722355, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0] 5
1.546631549848753


In [16]:
### Question 3

In [17]:
def featureQ3(datum, maxLen):
    # Implement 
    date = dateutil.parser.parse(datum['date_added'])
    return [1, len(datum['review_text']) / maxLen, int(date.weekday()), int(date.month)]

In [18]:
def Q3(dataset):
    # Implement (note MSE should be a *number*, not e.g. an array of length 1)
        maxLen = getMaxLen(dataset)
        X3 = []
        Y3 = []
        for datum in dataset:
            X3.append(featureQ3(datum, maxLen))
            Y3.append(datum['rating'])
        print(X3[1], Y3[2])
        reg = linear_model.LinearRegression(fit_intercept=False)
        reg.fit(X3,Y3)
   
        Yi = numpy.sum(numpy.asarray(X3) * reg.coef_, axis=1)
   
        MSE3 = sum((Yi - Y3)**2)/len(Y3)
        # theta = reg.coef_
        return X3, Y3, MSE3

In [19]:
X3, Y3, MSE3 = Q3(dataset)
print(MSE3)

[1, 0.10631902698168601, 2, 9] 5
1.5516353711453306


In [20]:
### Question 4

In [21]:
def Q4(dataset):
    n = len(dataset) // 2
    dataset_train, dataset_test = dataset[:n], dataset[n:]

    maxLen = getMaxLen(dataset_train)

    X2_train = [featureQ2(datum, maxLen) for datum in dataset_train]
    Y2_train = [datum['rating'] for datum in dataset_train]
    X2_test = [featureQ2(datum, maxLen) for datum in dataset_test]
    Y2_test = [datum['rating'] for datum in dataset_test]

    reg2 = linear_model.LinearRegression(fit_intercept=False)
    reg2.fit(X2_train, Y2_train)
    Y2i = numpy.sum(numpy.asarray(X2_test) * reg2.coef_, axis=1)
    test_mse2 = sum((Y2i - Y2_test)**2) / len(Y2_test)

    X3_train = [featureQ3(datum, maxLen) for datum in dataset_train]
    Y3_train = [datum['rating'] for datum in dataset_train]
    X3_test = [featureQ3(datum, maxLen) for datum in dataset_test]
    Y3_test = [datum['rating'] for datum in dataset_test]

    reg3 = linear_model.LinearRegression(fit_intercept=False)
    reg3.fit(X3_train, Y3_train)
    Y3i = numpy.sum(numpy.asarray(X3_test) * reg3.coef_, axis=1)
    test_mse3 = sum((Y3i - Y3_test)**2) / len(Y3_test)

    return test_mse2, test_mse3

In [22]:
dataset4 = dataset[:]
random.seed(0)
random.shuffle(dataset4)

test_mse2, test_mse3 = Q4(dataset4)
print(test_mse2, test_mse3)

1.626445367616787 1.6282919476176076


In [23]:
### Question 5

In [24]:
f = open("/Users/hrithik/Desktop/UC San Diego/Courses/UCSD CSE 258/Homework 1/beer_50000.json")
datasetB = []
for l in f:
    datasetB.append(eval(l))

In [25]:
datasetB[0]

{'review/appearance': 2.5,
 'beer/style': 'Hefeweizen',
 'review/palate': 1.5,
 'review/taste': 1.5,
 'beer/name': 'Sausa Weizen',
 'review/timeUnix': 1234817823,
 'beer/ABV': 5.0,
 'beer/beerId': '47986',
 'beer/brewerId': '10325',
 'review/timeStruct': {'isdst': 0,
  'mday': 16,
  'hour': 20,
  'min': 57,
  'sec': 3,
  'mon': 2,
  'year': 2009,
  'yday': 47,
  'wday': 0},
 'review/overall': 1.5,
 'review/text': 'A lot of foam. But a lot.\tIn the smell some banana, and then lactic and tart. Not a good start.\tQuite dark orange in color, with a lively carbonation (now visible, under the foam).\tAgain tending to lactic sourness.\tSame for the taste. With some yeast and banana.',
 'user/profileName': 'stcules',
 'review/aroma': 2.0}

In [26]:
def featureQ5(datum):
    # Implement
    x = [1, len(datum['review/text'])]
    y = int(datum['review/overall']>=4)
    return x, y

In [27]:
x, y = featureQ5(datasetB[0])
print(x, y)

[1, 262] 0


In [77]:
def Q5(dataset, feat_func):
    # Implement
    X5 = []
    Y5 = []
    for datum in dataset:
        x, y = feat_func(datum)
        X5.append(x)
        Y5.append(y)


    model = linear_model.LogisticRegression(class_weight='balanced',fit_intercept=False, max_iter=5000,)
    model.fit(X5, Y5)
    pred = model.predict(X5)
    # print(pred)
    TP = sum(numpy.logical_and(Y5, pred))
    TN = sum(numpy.logical_and(numpy.logical_not(Y5), numpy.logical_not(pred)))
    FP = sum(numpy.logical_and(numpy.logical_not(Y5), pred))
    FN = sum(numpy.logical_and(Y5, numpy.logical_not(pred)))

    TPR = TP/(TP+FN)
    TNR = TN/(TN+FP)

    BER = 1 - 0.5*(TPR + TNR)

    return TP, TN, FP, FN, BER

In [78]:
TP, TN, FP, FN, BER = Q5(datasetB, featureQ5)
print(TP, TN, FP, FN, BER)

14201 10503 5885 19411 0.4683031525957275


/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: inval

In [79]:
### Question 6

In [80]:
def Q6(dataset):
    # Compute the precision@K of your classifier for K ∈ {1, 100, 1000, 10000}

    X5 = []
    Y5 = []
    for datum in dataset:
        x, y = [1, len(datum['review/text'])], int(datum['review/overall']>=4)
        X5.append(x)
        Y5.append(y)

    model = linear_model.LogisticRegression(class_weight='balanced',fit_intercept=False)
    model.fit(X5, Y5)
    pred = model.decision_function(X5)
    # print(pred)
    score_labels = list(zip(pred, Y5))
    score_labels.sort(key=lambda pair: pair[0], reverse=True)

    return tuple(
        sum(label for _, label in score_labels[:k]) / k
        for k in [1, 100, 1000, 10000]
    )


In [81]:
k1, k2, k3, k4 = Q6(datasetB)
print(k1, k2, k3, k4 )

1.0 0.75 0.71 0.7147


/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: inval

In [82]:
### Question 7

In [83]:
datasetB[0]

{'review/appearance': 2.5,
 'beer/style': 'Hefeweizen',
 'review/palate': 1.5,
 'review/taste': 1.5,
 'beer/name': 'Sausa Weizen',
 'review/timeUnix': 1234817823,
 'beer/ABV': 5.0,
 'beer/beerId': '47986',
 'beer/brewerId': '10325',
 'review/timeStruct': {'isdst': 0,
  'mday': 16,
  'hour': 20,
  'min': 57,
  'sec': 3,
  'mon': 2,
  'year': 2009,
  'yday': 47,
  'wday': 0},
 'review/overall': 1.5,
 'review/text': 'A lot of foam. But a lot.\tIn the smell some banana, and then lactic and tart. Not a good start.\tQuite dark orange in color, with a lively carbonation (now visible, under the foam).\tAgain tending to lactic sourness.\tSame for the taste. With some yeast and banana.',
 'user/profileName': 'stcules',
 'review/aroma': 2.0}

In [84]:
def featureQ7(datum):
    x = [1, len(datum['review/text']), datum['review/appearance'], datum['review/palate'], datum['review/taste'], datum['review/aroma'], datum['beer/ABV']]
    y = int(datum['review/overall']>=4)
    return x, y

In [85]:
def testQ7():
    _, _, _, _, BER5 = Q5(datasetB, featureQ5)
    _, _, _, _, BER7 = Q5(datasetB, featureQ7)
    return BER5,BER7

BER5, BER7 = testQ7()
print(BER5, BER7)
print(((BER7-BER5)/BER5)*100)

/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: inval

0.4683031525957275 0.17549059692554336
-62.52628325202857


/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/hrithik/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
